# Citation Report Tools — full run

Exercises all **10 citation-report tools** in `akd_ext.tools.citation_report`, one by one,
in pipeline order. Each tool is an akd-ext `BaseTool`; we call it via `Tool().as_function()`
(the same path the MCP server uses).

There is **no extraction agent here** — that LLM step lives in the agent layer. Between
`get_pdf_text` and `save_report` we substitute a **dummy report** to stand in for what the
Citation Analyzer agent would produce.

Everything is written under `version="nbtest"` so the last cell cleans it all up.

**Required env vars:** `SEMANTIC_SCHOLAR_API_KEY`, `FM_S3_BUCKET`, `FM_S3_PREFIX`,
`AWS_REGION`, `UNPAYWALL_EMAIL`, and AWS credentials (`AWS_ACCESS_KEY_ID` /
`AWS_SECRET_ACCESS_KEY` / `AWS_SESSION_TOKEN`). Set them in the environment or a `.env`
loaded before running.

In [1]:
import asyncio, json, threading, urllib.request
from concurrent.futures import Future

# Optional: load a .env if present (python-dotenv). Skip if you export vars another way.
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

from akd_ext.tools import (
    ResolvePaperTool, GetCitationsTool, EnrichCitationsTool, DownloadPdfsTool,
    GetSeedTextTool, GetPdfTextTool, SaveReportTool, GetReportTool,
    ConsolidateReportsTool, GetDownloadLinkTool,
)
from akd_ext.tools.citation_report._lib.env import get_cache

cache = get_cache()
VERSION = "nbtest"          # isolate everything this notebook writes
SEED = "2310.18660"         # Prithvi geospatial FM paper (arXiv id)
LIMIT = 3                   # keep the run small/fast

# Each tool is a BaseTool; .as_function(mode='python') gives an async callable returning a dict.
# Jupyter already runs an event loop, so run each coroutine in a fresh loop on its own thread.
def _run_async(coro):
    box = Future()
    def worker():
        loop = asyncio.new_event_loop()
        try:
            box.set_result(loop.run_until_complete(coro))
        except Exception as e:
            box.set_exception(e)
        finally:
            loop.close()
    t = threading.Thread(target=worker)
    t.start(); t.join()
    return box.result()

def call(ToolCls, **kwargs):
    return _run_async(ToolCls().as_function(mode="python")(**kwargs))

print("cache enabled:", cache.enabled, "| creds:", cache.check_credentials())

/rhome/sawale/akd_with_care/akd-ext/akd-ext/.venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


cache enabled: True | creds: {'ok': True}


## 1. ResolvePaperTool
Resolve the seed paper reference → canonical Semantic Scholar `paper_id` (+ abstract).

In [2]:
r = call(ResolvePaperTool, user_input=SEED)
seed_paper_id = r["paper_id"]
print("status:", r["status"], "| kind:", r["input_kind"])
print("paper_id:", seed_paper_id)
print("title   :", r["title"])
print("cites   :", r["citation_count"], "| abstract chars:", len(r["abstract"]))

status: ok | kind: id/url
paper_id: e966bd21790a6c6d0a07b4917325767451c9e23e
title   : Foundation Models for Generalist Geospatial Artificial Intelligence
cites   : 198 | abstract chars: 1894


## 2. GetSeedTextTool  (+ dummy Target Profiler)
Get the seed paper's text (full PDF, abstract fallback). The real workflow feeds this to a
**Target Profiler agent**; here we fake a `target_spec`.

In [3]:
st = call(GetSeedTextTool, external_ids=r["external_ids"], title=r["title"],
          abstract=r["abstract"], max_chars=30000)
print("status:", st["status"], "| source:", st["source"], "| pages:", st["page_count"],
      "| chars:", st["char_count"])

# --- dummy Target Profiler (the agent's job in the real workflow) ---
target_spec = {"name": "Prithvi", "aliases": ["Prithvi-100M", "Prithvi-300M", "IBM Prithvi"],
               "domain": "geospatial foundation model",
               "what_to_look_for": "fine-tuning / benchmarking / feature-extraction of Prithvi"}
print("target_spec:", target_spec["name"], "| aliases:", target_spec["aliases"])

status: ok | source: pdf | pages: 26 | chars: 30000
target_spec: Prithvi | aliases: ['Prithvi-100M', 'Prithvi-300M', 'IBM Prithvi']


## 3. GetCitationsTool — peek (cache status, no fetch)
`peek_only=True` reports cache status so you can warn before a slow fresh fetch.

In [4]:
pk = call(GetCitationsTool, paper_id=seed_paper_id, peek_only=True,
          citation_count=r["citation_count"])
print("cache:", pk["cache"])

cache: {'cached': True, 'complete': False, 'fetched_at': '2026-05-31T06:11:11Z', 'age_days': 0.82, 'record_count': 10, 'estimated_fresh_seconds': 6}


## 4. GetCitationsTool — fetch (limited)
Serves the cache if it satisfies the request, else fetches fresh. Capped at `LIMIT`.

In [5]:
gc = call(GetCitationsTool, paper_id=seed_paper_id, use_cache=True, paper_limit=LIMIT)
records = gc["records"]
print("served:", gc["served"], "| from_cache:", gc["from_cache"], "| count:", gc["record_count"])
print("citations_s3_uri:", gc["citations_s3_uri"])
for rec in records:
    cp = rec["citingPaper"]
    print("  -", (cp.get("title") or "")[:55], "| ids:", cp.get("externalIds"))

served: cache_sliced | from_cache: True | count: 3
citations_s3_uri: s3://akd-citation-report-agent-cache/fm-report-cache/citations/e966bd21790a6c6d0a07b4917325767451c9e23e/all_citations.json
  - FLORO: A Multimodal Geospatial Foundation Model for Eco | ids: {'ArXiv': '2605.28174', 'CorpusId': 288701696}
  - SpectralEarth-FM: Bringing Hyperspectral Imagery into M | ids: {'ArXiv': '2605.21075', 'CorpusId': 288653239}
  - Mini-JEPA Foundation Model Fleet Enables Agentic Hydrol | ids: {'ArXiv': '2605.14120', 'CorpusId': 288253369}


## 5. EnrichCitationsTool
Validate/fill external ids. Records that already have ArXiv/DOI are skipped (no API call).

In [6]:
en = call(EnrichCitationsTool, records=records)
records = en["records"]
print("enriched:", en["enriched"], "| already_had:", en["already_had"], "| filled:", en["filled"],
      "| no_id:", en["flagged_no_id"], "| no_oa:", en["flagged_no_oa"])

enriched: 3 | already_had: 3 | filled: 0 | no_id: 0 | no_oa: 0


## 6. DownloadPdfsTool
Download PDFs into the S3 pdf cache (cache-first). Each record gets a `pdf_cache_key`.

In [7]:
dl = call(DownloadPdfsTool, records=records)
records = dl["records"]
print("downloaded:", dl["downloaded"], "| from_cache:", dl["from_cache"],
      "| failed:", dl["failed"], "| no_id:", dl["no_id"])
for rec in records:
    cp = rec["citingPaper"]
    print("  -", cp.get("pdf_status"), "|", cp.get("pdf_cache_key"))

ready = [rec["citingPaper"]["pdf_cache_key"] for rec in records
         if rec["citingPaper"].get("pdf_status") in ("cached", "downloaded")]
print("\nPDFs ready to extract:", ready)

downloaded: 0 | from_cache: 3 | failed: 0 | no_id: 0
  - cached | 2605.28174
  - cached | 2605.21075
  - cached | 2605.14120

PDFs ready to extract: ['2605.28174', '2605.21075', '2605.14120']


## 7. GetPdfTextTool  +  (dummy extraction)  +  8. SaveReportTool
Per paper: pull the PDF text from S3, then — **standing in for the Citation Analyzer agent** —
build a **dummy report** and save it. Replace `make_dummy_report` with real agent output.

In [8]:
def make_dummy_report(text, target_spec):
    """Stand-in for the Citation Analyzer agent's structured output."""
    mentions = target_spec["name"].lower() in (text or "").lower()
    usage = "benchmark" if mentions else "mention_only"
    return {
        "report": {
            "usage_type": usage,
            "usage_summary": "DUMMY — generated by the notebook, not a real extraction.",
            "metrics": {
                "usage_type_primary": usage,
                "evidence_quality": {"confidence": 0.5},
                "results_signal": {"reported_improvement": "unknown"},
                "downstream_tasks": [],
            },
        },
        "evidence": [{"quote": "(dummy)", "page_hint": 1}] if mentions else [],
    }

saved_keys = []
for key in ready:
    txt = call(GetPdfTextTool, pdf_cache_key=key, max_chars=20000)
    print(f"get_pdf_text {key}: status={txt['status']} pages={txt['page_count']} chars={txt['char_count']}")
    if txt["status"] != "ok":
        continue
    # --- dummy extraction (the agent's LLM call would go here) ---
    report = make_dummy_report(txt["text"], target_spec)
    sv = call(SaveReportTool, cache_key=key, report=report,
              seed_paper_id=seed_paper_id, version=VERSION, paper_id=key)
    print(f"  save_report: {sv['status']} -> {sv['s3_uri']}")
    saved_keys.append(key)

print("\nsaved:", saved_keys)

get_pdf_text 2605.28174: status=ok pages=29 chars=20018
  save_report: ok -> s3://akd-citation-report-agent-cache/fm-report-cache/reports/e966bd21790a6c6d0a07b4917325767451c9e23e/nbtest/2605.28174.json
get_pdf_text 2605.21075: status=ok pages=21 chars=20018
  save_report: ok -> s3://akd-citation-report-agent-cache/fm-report-cache/reports/e966bd21790a6c6d0a07b4917325767451c9e23e/nbtest/2605.21075.json
get_pdf_text 2605.14120: status=ok pages=40 chars=20018
  save_report: ok -> s3://akd-citation-report-agent-cache/fm-report-cache/reports/e966bd21790a6c6d0a07b4917325767451c9e23e/nbtest/2605.14120.json

saved: ['2605.28174', '2605.21075', '2605.14120']


## 9. GetReportTool
Read one report back (cache-first helper — lets the agent skip re-analysis).

In [9]:
if saved_keys:
    g = call(GetReportTool, cache_key=saved_keys[0], seed_paper_id=seed_paper_id, version=VERSION)
    print("exists:", g["exists"])
    print(json.dumps(g["report"], indent=2)[:600])
else:
    print("no saved reports to read")

exists: True
{
  "report": {
    "usage_type": "mention_only",
    "usage_summary": "DUMMY \u2014 generated by the notebook, not a real extraction.",
    "metrics": {
      "usage_type_primary": "mention_only",
      "evidence_quality": {
        "confidence": 0.5
      },
      "results_signal": {
        "reported_improvement": "unknown"
      },
      "downstream_tasks": []
    }
  },
  "evidence": [],
  "_meta": {
    "saved_at": "2026-06-01T01:57:01Z",
    "paper_id": "2605.28174",
    "cache_key": "2605.28174",
    "seed_paper_id": "e966bd21790a6c6d0a07b4917325767451c9e23e",
    "version": "nbtest"
 


## 10. ConsolidateReportsTool
Merge the saved per-paper reports into one master JSON (written to S3). By default each
entry embeds the full report + evidence (`include_full=True`).

In [10]:
co = call(ConsolidateReportsTool, seed_paper_id=seed_paper_id, cache_keys=saved_keys, version=VERSION)
print("status:", co["status"], "| total:", co["total_papers"], "| missing:", co["missing"])
print("counts:", co["counts_by_usage_type"])
print("master:", co["master_s3_uri"])
master_uri = co["master_s3_uri"]
if co["master"]["papers"]:
    p0 = co["master"]["papers"][0]
    print("first entry embeds full report:", "report" in p0, "| evidence:", "evidence" in p0)

status: ok | total: 3 | missing: []
counts: {'mention_only': 2, 'benchmark': 1}
master: s3://akd-citation-report-agent-cache/fm-report-cache/reports/e966bd21790a6c6d0a07b4917325767451c9e23e/nbtest/_master.json
first entry embeds full report: True | evidence: True


## 11. GetDownloadLinkTool  (report + citations)
Presign the master report JSON and the citation list. The presigned URL is the only
browser-openable way to download from the private bucket — we verify by fetching it with no
AWS creds.

In [11]:
# report download link
lk = call(GetDownloadLinkTool, s3_uri_or_key=master_uri, expires_days=1)
print("report link :", lk["status"], "| expires:", lk["expires_at"])
with urllib.request.urlopen(lk["url"], timeout=30) as resp:
    body = resp.read()
print("  downloaded master (no creds): total_papers =", json.loads(body)["total_papers"], "| bytes =", len(body))

# citations download link
if gc["citations_s3_uri"]:
    lk2 = call(GetDownloadLinkTool, s3_uri_or_key=gc["citations_s3_uri"], expires_days=1)
    print("citations link:", lk2["status"])

report link : ok | expires: 2026-06-02T01:57:13Z
  downloaded master (no creds): total_papers = 3 | bytes = 3434
citations link: ok


## Cleanup
Delete everything written under `version="nbtest"` (per-paper reports + master). PDFs are
left in the cache (real, reusable); the citation-list cache is also left in place.

In [12]:
deleted = 0
for key in saved_keys:
    cache.client.delete_object(Bucket=cache.bucket,
                               Key=cache._agent_report_key(seed_paper_id, VERSION, key))
    deleted += 1
cache.client.delete_object(Bucket=cache.bucket,
                           Key=cache.master_report_key(seed_paper_id, VERSION))
deleted += 1
print("deleted", deleted, "nbtest objects")

left = cache.client.list_objects_v2(
    Bucket=cache.bucket,
    Prefix=cache._key("reports", seed_paper_id, VERSION),
).get("KeyCount", 0)
print("remaining nbtest keys:", left)

deleted 4 nbtest objects
remaining nbtest keys: 0
